In [1]:
from argparse import Namespace

In [2]:
import pandas as pd

In [3]:
import torch

In [ ]:
from transformer_lens.model_bridge import TransformerBridge

In [5]:
#from eap.graph import Graph
#from eap import graph
from eap.utils import tokenize_plus

In [6]:
from ablation_utils.utils import make_hooks

In [7]:
from gc import collect
from torch.cuda import empty_cache

In [8]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

In [9]:
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [10]:
DEVICE='cuda:7'

In [6]:
# import importlib
# importlib.reload(graph)

In [ ]:
# collect()
# empty_cache()

In [ ]:
if "WORK" not in os.environ:
    os.environ["WORK"] = '..'
DATA_DIR = os.environ["WORK"] + '/RW_functionalities_results'

In [ ]:
subnodes_df = pd.read_csv(DATA_DIR + '/subnodes.csv')

## Inspecting df

In [11]:
subnodes_df.head(40)

,Unnamed: 0,name,layer,node_name,subnode_index,score
0,0,input,0,input,NaN,0.000000e+00
1,1,a0.h0,0,a0.h0,NaN,0.000000e+00
2,2,a0.h1,0,a0.h1,NaN,0.000000e+00
3,3,a0.h2,0,a0.h2,NaN,0.000000e+00
4,4,a0.h3,0,a0.h3,NaN,0.000000e+00
5,5,a0.h4,0,a0.h4,NaN,0.000000e+00
6,6,a0.h5,0,a0.h5,NaN,0.000000e+00
7,7,a0.h6,0,a0.h6,NaN,0.000000e+00
8,8,a0.h7,0,a0.h7,NaN,0.000000e+00
9,9,a0.h8,0,a0.h8,NaN,0.000000e+00


In [12]:
sorted_df = subnodes_df.sort_values("score", ascending=False).head(49)
print(sorted_df)

      Unnamed: 0      name  layer node_name  subnode_index     score
1549        1549  m31.8581     31       m31         8581.0  1.510321
1405        1405  m30.9996     30       m30         9996.0  1.103545
527          527       m12     12       m12            NaN  0.726136
1424        1424   a31.h12     31   a31.h12            NaN  0.663149
1339        1339   a30.h10     30   a30.h10            NaN  0.620403
1320        1320  m29.7261     29       m29         7261.0  0.571579
685          685       m16     16       m16            NaN  0.542899
1444        1444       m31     31       m31            NaN  0.520416
981          981       m23     23       m23            NaN  0.467276
1363        1363   m30.732     30       m30          732.0  0.456894
1345        1345   a30.h16     30   a30.h16            NaN  0.377761
1043        1043  m24.7935     24       m24         7935.0  0.374789
1306        1306  m29.4131     29       m29         4131.0  0.373894
566          566       m13     13 

## Getting the rows we care about

In [12]:
weakening_df = subnodes_df[subnodes_df["subnode_index"].notna()].sort_values("score", ascending=False)
print(weakening_df.shape)

(526, 6)


In [13]:
sorted_filtered_df = weakening_df[weakening_df["score"]>0].sort_values("score", ascending=False)
print(sorted_filtered_df.shape)

(246, 6)


## Inspecting again

In [26]:
print(sorted_filtered_df.head(40))

      Unnamed: 0       name  layer node_name  subnode_index     score
1549        1549   m31.8581     31       m31         8581.0  1.510321
1405        1405   m30.9996     30       m30         9996.0  1.103545
1320        1320   m29.7261     29       m29         7261.0  0.571579
1363        1363    m30.732     30       m30          732.0  0.456894
1043        1043   m24.7935     24       m24         7935.0  0.374789
1306        1306   m29.4131     29       m29         4131.0  0.373894
1489        1489   m31.3498     31       m31         3498.0  0.193198
1377        1377   m30.3705     30       m30         3705.0  0.114631
1506        1506   m31.4568     31       m31         4568.0  0.114614
1148        1148   m26.5558     26       m26         5558.0  0.085032
612          612   m14.4516     14       m14         4516.0  0.066492
1253        1253   m28.6703     28       m28         6703.0  0.039890
1524        1524   m31.5973     31       m31         5973.0  0.037797
1551        1551   m

In [27]:
print(sorted_filtered_df.tail(40))

      Unnamed: 0       name  layer node_name  subnode_index     score
382          382      m8.22      8        m8           22.0  0.000156
861          861   m20.1821     20       m20         1821.0  0.000153
208          208     m4.382      4        m4          382.0  0.000143
260          260    m5.6293      5        m5         6293.0  0.000141
1481        1481   m31.2938     31       m31         2938.0  0.000140
1519        1519   m31.5751     31       m31         5751.0  0.000125
1252        1252   m28.6431     28       m28         6431.0  0.000120
129          129    m2.3729      2        m2         3729.0  0.000117
1326        1326  m29.10283     29       m29        10283.0  0.000110
1567        1567  m31.10439     31       m31        10439.0  0.000108
128          128     m2.211      2        m2          211.0  0.000108
1080        1080    m25.782     25       m25          782.0  0.000107
1309        1309   m29.5237     29       m29         5237.0  0.000105
689          689  m1

In [25]:
weakening_df.tail(40)

,Unnamed: 0,name,layer,node_name,subnode_index,score
490,490,m11.1068,11,m11,1068.0,-0.009033
1466,1466,m31.1710,31,m31,1710.0,-0.009061
1366,1366,m30.1511,30,m30,1511.0,-0.009455
1084,1084,m25.2489,25,m25,2489.0,-0.009744
1410,1410,m30.10850,30,m30,10850.0,-0.009938
985,985,m23.513,23,m23,513.0,-0.010247
1404,1404,m30.9656,30,m30,9656.0,-0.010870
1037,1037,m24.2931,24,m24,2931.0,-0.010933
1448,1448,m31.287,31,m31,287.0,-0.011020
1034,1034,m24.619,24,m24,619.0,-0.011640


## Evaluating via ablations

### General

In [ ]:
model = TransformerBridge.boot_transformers(#with predefined neurons, our ablation code is implementation-invariant
        'allenai/OLMo-7B-0424-hf',
        device=DEVICE,
)
#needed?:
# model.enable_compatibility_mode(
#         no_processing=True
# )

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loaded pretrained model allenai/OLMo-7B-0424-hf into HookedTransformer


In [15]:
sequence = "Yesterday (21 December) the Government announced a package of support for hospitality and leisure businesses that are losing trade because of the O"

In [16]:
tokens, attention_mask, input_lengths, n_pos = tokenize_plus(model, [sequence])

In [17]:
logits = model(tokens, attention_mask=attention_mask)
score = logits[...,-1,model.to_single_token('mic')].item()
print(score)

21.09280776977539


In [26]:
def run_ablations(args:Namespace, df:pd.DataFrame, score_list:list|None=None, reverse:bool=False):
    if score_list is None:
        score_list = []
    hook_list = []
    if reverse:
        iterator = df[::-1].iterrows()
    else:
        iterator = df.iterrows()
    for i,row in iterator:
        hook_list.extend(
            make_hooks(
                args,
                layer=row["layer"], neuron=int(row["subnode_index"]),
                mean_value=0.0,
            )
        )
        with model.hooks(hook_list):
            logits = model(tokens, attention_mask=attention_mask)
            score = logits[...,-1,model.to_single_token('mic')].item()
            score_list.append(score)
            print(len(score_list), ', ', score)
    return score_list

### Conditional ablations

In [20]:
conditional_args = Namespace(
    gate='-', post='+',
    activation_location='mlp.hook_post',
    intervention_type='zero_ablation',
)

In [21]:
conditional_score_list = run_ablations(conditional_args, weakening_df)

0 ,  21.100914001464844
1 ,  19.696533203125
2 ,  18.707666397094727
3 ,  17.81782341003418
4 ,  16.941997528076172
5 ,  15.728752136230469
6 ,  15.728752136230469
7 ,  15.550999641418457
8 ,  15.550999641418457
9 ,  14.758160591125488
10 ,  14.822168350219727
11 ,  12.08298110961914
12 ,  12.08298110961914
13 ,  12.07059097290039
14 ,  9.4954833984375
15 ,  9.4954833984375
16 ,  9.234922409057617
17 ,  9.234922409057617
18 ,  9.20826244354248
19 ,  9.20826244354248
20 ,  9.20826244354248
21 ,  9.20826244354248
22 ,  9.20826244354248
23 ,  9.213786125183105
24 ,  9.215871810913086
25 ,  9.215871810913086
26 ,  9.215871810913086
27 ,  9.215871810913086
28 ,  9.206403732299805
29 ,  9.214556694030762
30 ,  9.210439682006836
31 ,  9.208571434020996
32 ,  9.208571434020996
33 ,  9.208826065063477
34 ,  9.22209358215332
35 ,  9.22209358215332
36 ,  9.22011661529541
37 ,  9.22011661529541
38 ,  9.220125198364258
39 ,  9.220125198364258
40 ,  9.195072174072266
41 ,  9.192594528198242
42 ,  9.

### How about non-conditional ablations?

In [23]:
unconditional_args = Namespace(
    gate=None, post=None,
    activation_location='mlp.hook_post',
    intervention_type='zero_ablation',
)

In [24]:
unconditional_score_list = run_ablations(unconditional_args, weakening_df)

1 ,  21.100914001464844
2 ,  19.676048278808594
3 ,  18.67130470275879
4 ,  17.786930084228516
5 ,  16.908519744873047
6 ,  15.697864532470703
7 ,  15.514573097229004
8 ,  15.350500106811523
9 ,  15.177751541137695
10 ,  14.407808303833008
11 ,  14.552196502685547
12 ,  12.199928283691406
13 ,  12.202348709106445
14 ,  12.191187858581543
15 ,  9.844694137573242
16 ,  9.841757774353027
17 ,  9.552042007446289
18 ,  9.1091890335083
19 ,  9.116903305053711
20 ,  9.107412338256836
21 ,  9.107223510742188
22 ,  9.129350662231445
23 ,  9.126296997070312
24 ,  9.133541107177734
25 ,  9.134151458740234
26 ,  9.129873275756836
27 ,  9.130594253540039
28 ,  9.129091262817383
29 ,  9.131183624267578
30 ,  9.147056579589844
31 ,  9.140364646911621
32 ,  9.126411437988281
33 ,  9.113723754882812
34 ,  9.112071990966797
35 ,  9.145425796508789
36 ,  9.177021026611328
37 ,  9.139070510864258
38 ,  9.018017768859863
39 ,  9.023193359375
40 ,  9.022941589355469
41 ,  8.991870880126953
42 ,  9.107089996

### How about reversing the order?

In [27]:
reversed_unconditional_score_list = run_ablations(unconditional_args, weakening_df, reverse=True)

1 ,  21.187631607055664
2 ,  21.18760871887207
3 ,  21.32521629333496
4 ,  21.392553329467773
5 ,  21.497812271118164
6 ,  20.80942726135254
7 ,  20.47566795349121
8 ,  20.83644676208496
9 ,  20.869461059570312
10 ,  20.927759170532227
11 ,  20.96639633178711
12 ,  21.03947639465332
13 ,  20.996511459350586
14 ,  21.02628517150879
15 ,  20.87240219116211
16 ,  20.888755798339844
17 ,  20.930601119995117
18 ,  20.927644729614258
19 ,  20.912261962890625
20 ,  20.96051597595215
21 ,  20.47476577758789
22 ,  20.479270935058594
23 ,  20.468238830566406
24 ,  20.49325942993164
25 ,  20.487550735473633
26 ,  20.325393676757812
27 ,  20.309423446655273
28 ,  20.107038497924805
29 ,  20.128925323486328
30 ,  20.140384674072266
31 ,  20.217620849609375
32 ,  20.22705078125
33 ,  20.231624603271484
34 ,  20.226749420166016
35 ,  20.253429412841797
36 ,  20.254255294799805
37 ,  20.284706115722656
38 ,  20.285686492919922
39 ,  20.29203987121582
40 ,  20.294017791748047
41 ,  20.286739349365234
4

### Trying other combinations (unconditional)

The first 15 neurons and the last one seem relevant.

In [31]:
hook_list = []
layer, neuron = weakening_df.iloc[-1]["layer"], int(weakening_df.iloc[-1]["subnode_index"])
print(layer, neuron)
hook_list.extend(
        make_hooks(
            unconditional_args,
            layer=layer, neuron=neuron,
            mean_value=0.0,
        )
    )
counter=1
for i, row in weakening_df.iterrows():
    hook_list.extend(
        make_hooks(
            unconditional_args,
            layer=row["layer"], neuron=int(row["subnode_index"]),
            mean_value=0.0,
        )
    )
    counter+=1
    with model.hooks(hook_list):
        logits = model(tokens, attention_mask=attention_mask)
        score = logits[...,-1,model.to_single_token('mic')].item()
    print(counter, score)

19 9428
2 21.19553565979004
3 19.778331756591797
4 18.690719604492188
5 17.854339599609375
6 17.01710319519043
7 15.684441566467285
8 15.494027137756348
9 15.326513290405273
10 15.146973609924316
11 14.218110084533691
12 14.32314682006836
13 10.61269760131836
14 10.616426467895508
15 10.605169296264648
16 7.987966537475586
17 7.986255168914795
18 7.765355587005615
19 7.287669658660889
20 7.287300109863281
21 7.282834053039551
22 7.282846450805664
23 7.289073467254639
24 7.2883429527282715
25 7.289684295654297
26 7.288772106170654
27 7.287337779998779
28 7.28761625289917
29 7.287458419799805
30 7.287285804748535
31 7.2848711013793945
32 7.278609752655029
33 7.272932052612305
34 7.267281532287598
35 7.271599769592285
36 7.283195972442627
37 7.2919816970825195
38 7.291751861572266
39 7.257363319396973
40 7.262372970581055
41 7.262213706970215
42 7.2526068687438965
43 7.255599021911621
44 7.246738433837891
45 7.246380805969238
46 7.253517150878906
47 7.241196632385254
48 7.240120887756348


We get significant drops after ablating 507 or 513 neurons, so neurons number 505 and 511 seem relevant.

In [32]:
hook_list = []
extra_list = [-1, 505,511]
for i in extra_list:
    layer, neuron = weakening_df.iloc[i]["layer"], int(weakening_df.iloc[i]["subnode_index"])
    print(layer, neuron)
    hook_list.extend(
            make_hooks(
                unconditional_args,
                layer=layer, neuron=neuron,
                mean_value=0.0,
            )
        )
counter=len(extra_list)
for i, row in weakening_df.iterrows():
    hook_list.extend(
        make_hooks(
            unconditional_args,
            layer=row["layer"], neuron=int(row["subnode_index"]),
            mean_value=0.0,
        )
    )
    counter+=1
    with model.hooks(hook_list):
        logits = model(tokens, attention_mask=attention_mask)
        score = logits[...,-1,model.to_single_token('mic')].item()
    print(counter, score)

19 9428
18 6799
28 5325
4 20.84912109375
5 19.449995040893555
6 18.406784057617188
7 17.627052307128906
8 16.75486183166504
9 15.319100379943848
10 15.122270584106445
11 14.940174102783203
12 14.754110336303711
13 13.42409896850586
14 13.577224731445312
15 8.380317687988281
16 8.383272171020508
17 8.37763786315918
18 6.944849014282227
19 6.94462776184082
20 6.846426010131836
21 4.99641752243042
22 4.991631031036377
23 4.994928359985352
24 4.994936943054199
25 4.999763488769531
26 4.999791145324707
27 5.041248321533203
28 5.03647518157959
29 5.036261558532715
30 5.03632926940918
31 5.036343097686768
32 5.05977201461792
33 5.03472375869751
34 5.026397705078125
35 4.955720901489258
36 4.97201681137085
37 4.88156795501709
38 4.971934795379639
39 4.998709678649902
40 5.045924186706543
41 4.52802848815918
42 4.536159992218018
43 4.53614616394043
44 4.378337860107422
45 4.447990417480469
46 4.392324447631836
47 4.392243385314941
48 4.515331745147705
49 4.390039920806885
50 4.389725208282471
5

Sharp drop after 70 neurons, so neuron 70-len(extra_list)-1=66 seems relevant.

In [54]:
def nice_ablation(extra_list):
    hook_list = []
    for i in extra_list:
        layer, neuron = weakening_df.iloc[i]["layer"], int(weakening_df.iloc[i]["subnode_index"])
        print(layer, neuron)
        hook_list.extend(
                make_hooks(
                    unconditional_args,
                    layer=layer, neuron=neuron,
                    mean_value=0.0,
                )
            )
    counter=len(extra_list)
    with model.hooks(hook_list):
        logits = model(tokens, attention_mask=attention_mask)
        score = logits[...,-1,model.to_single_token('mic')].item()
    print(f'ablating only the {counter} extra neurons yields score {score}')
    for i, row in weakening_df.iterrows():
        hook_list.extend(
            make_hooks(
                unconditional_args,
                layer=row["layer"], neuron=int(row["subnode_index"]),
                mean_value=0.0,
            )
        )
        counter+=1
        with model.hooks(hook_list):
            logits = model(tokens, attention_mask=attention_mask)
            score = logits[...,-1,model.to_single_token('mic')].item()
        print(f'ablating {counter} neurons, last of which is {row["layer"], int(row["subnode_index"])}, yields score {score}')
        if counter>len(extra_list)+extra_list[-1]:
            break

In [61]:
extra_list = [-1, 505, 511, 66]
nice_ablation(extra_list)

19 9428
18 6799
28 5325
10 10026
ablating only the 4 extra neurons yields score 20.571409225463867
ablating 5 neurons, last of which is (31, 8581), yields score 20.57946014404297
ablating 6 neurons, last of which is (30, 9996), yields score 19.184572219848633
ablating 7 neurons, last of which is (29, 7261), yields score 18.168004989624023
ablating 8 neurons, last of which is (30, 732), yields score 17.295026779174805
ablating 9 neurons, last of which is (24, 7935), yields score 16.386899948120117
ablating 10 neurons, last of which is (29, 4131), yields score 14.987414360046387
ablating 11 neurons, last of which is (31, 3498), yields score 14.792738914489746
ablating 12 neurons, last of which is (30, 3705), yields score 14.615364074707031
ablating 13 neurons, last of which is (31, 4568), yields score 14.430747985839844
ablating 14 neurons, last of which is (26, 5558), yields score 12.576786041259766
ablating 15 neurons, last of which is (14, 4516), yields score 12.646888732910156
ablati

Let's also ablate neuron 22-5=17

In [62]:
extra_list.append(17)
nice_ablation(extra_list)

19 9428
18 6799
28 5325
10 10026
25 4595
ablating only the 5 extra neurons yields score 20.64203453063965
ablating 6 neurons, last of which is (31, 8581), yields score 20.64957046508789
ablating 7 neurons, last of which is (30, 9996), yields score 19.18897247314453
ablating 8 neurons, last of which is (29, 7261), yields score 18.296916961669922
ablating 9 neurons, last of which is (30, 732), yields score 17.32617950439453
ablating 10 neurons, last of which is (24, 7935), yields score 16.39739990234375
ablating 11 neurons, last of which is (29, 4131), yields score 14.708066940307617
ablating 12 neurons, last of which is (31, 3498), yields score 14.509256362915039
ablating 13 neurons, last of which is (30, 3705), yields score 14.315394401550293
ablating 14 neurons, last of which is (31, 4568), yields score 14.135855674743652
ablating 15 neurons, last of which is (26, 5558), yields score 8.433174133300781
ablating 16 neurons, last of which is (14, 4516), yields score 7.904938697814941
abl

Finally (?), let's ablate neurons 17-6=11 and 15-6=9

In [63]:
extra_list.extend([11,9])
nice_ablation(extra_list)

19 9428
18 6799
28 5325
10 10026
25 4595
28 6703
26 5558
ablating only the 7 extra neurons yields score 19.36339569091797
ablating 8 neurons, last of which is (31, 8581), yields score 19.365585327148438
ablating 9 neurons, last of which is (30, 9996), yields score 16.88190460205078
ablating 10 neurons, last of which is (29, 7261), yields score 16.2645320892334
ablating 11 neurons, last of which is (30, 732), yields score 15.44491195678711
ablating 12 neurons, last of which is (24, 7935), yields score 14.512582778930664
ablating 13 neurons, last of which is (29, 4131), yields score 4.585321426391602
ablating 14 neurons, last of which is (31, 3498), yields score 4.575006484985352
ablating 15 neurons, last of which is (30, 3705), yields score 4.512831687927246
ablating 16 neurons, last of which is (31, 4568), yields score 4.495296478271484
ablating 17 neurons, last of which is (26, 5558), yields score 4.495296478271484


In [64]:
extra_list.append(5)
nice_ablation(extra_list)

19 9428
18 6799
28 5325
10 10026
25 4595
28 6703
26 5558
29 4131
ablating only the 8 extra neurons yields score 13.105241775512695
ablating 9 neurons, last of which is (31, 8581), yields score 13.032252311706543
ablating 10 neurons, last of which is (30, 9996), yields score 10.754203796386719
ablating 11 neurons, last of which is (29, 7261), yields score 1.5392382144927979
ablating 12 neurons, last of which is (30, 732), yields score 1.6689170598983765
ablating 13 neurons, last of which is (24, 7935), yields score 4.585321426391602
ablating 14 neurons, last of which is (29, 4131), yields score 4.585321426391602


In [65]:
extra_list.append(2)
nice_ablation(extra_list)

19 9428
18 6799
28 5325
10 10026
25 4595
28 6703
26 5558
29 4131
29 7261
ablating only the 9 extra neurons yields score 0.53655606508255
ablating 10 neurons, last of which is (31, 8581), yields score 0.7168054580688477
ablating 11 neurons, last of which is (30, 9996), yields score 1.5392382144927979
ablating 12 neurons, last of which is (29, 7261), yields score 1.5392382144927979


In [66]:
print(extra_list)

[-1, 505, 511, 66, 17, 11, 9, 5, 2]


### Top-down variable selection from this list

In [69]:
def top_down_selection(extra_list):
    scores = []
    for tested_index in extra_list:
        tested_layer, tested_neuron = weakening_df.iloc[tested_index]["layer"], int(weakening_df.iloc[tested_index]["subnode_index"])
        print("testing", tested_layer, tested_neuron)
        hook_list = []
        for other_index in extra_list:
            if other_index==tested_index:
                continue
            other_layer, other_neuron = weakening_df.iloc[other_index]["layer"], int(weakening_df.iloc[other_index]["subnode_index"])
            hook_list.extend(
                    make_hooks(
                        unconditional_args,
                        layer=other_layer, neuron=other_neuron,
                        mean_value=0.0,
                    )
                )
        with model.hooks(hook_list):
            logits = model(tokens, attention_mask=attention_mask)
            score = logits[...,-1,model.to_single_token('mic')].item()
        scores.append(score)
        print(f'ablating all relevant neurons except the tested one yields score {score}')
    return scores

In [70]:
scores = top_down_selection(extra_list)

testing 19 9428
ablating all relevant neurons except the tested one yields score 2.0624759197235107
testing 18 6799
ablating all relevant neurons except the tested one yields score 1.0580263137817383
testing 28 5325
ablating all relevant neurons except the tested one yields score 1.1810179948806763
testing 10 10026
ablating all relevant neurons except the tested one yields score 1.9695978164672852
testing 25 4595
ablating all relevant neurons except the tested one yields score 3.1584556102752686
testing 28 6703
ablating all relevant neurons except the tested one yields score 6.585380554199219
testing 26 5558
ablating all relevant neurons except the tested one yields score 6.343113422393799
testing 29 4131
ablating all relevant neurons except the tested one yields score 18.503150939941406
testing 29 7261
ablating all relevant neurons except the tested one yields score 13.105241775512695


In [73]:
new_extra_list = extra_list[:1]+extra_list[2:]
print(new_extra_list)

[-1, 511, 66, 17, 11, 9, 5, 2]


In [74]:
scores = top_down_selection(new_extra_list)

testing 19 9428
ablating all relevant neurons except the tested one yields score 3.7353315353393555
testing 28 5325
ablating all relevant neurons except the tested one yields score 2.1548662185668945
testing 10 10026
ablating all relevant neurons except the tested one yields score 3.834749221801758
testing 25 4595
ablating all relevant neurons except the tested one yields score 5.270565032958984
testing 28 6703
ablating all relevant neurons except the tested one yields score 7.596182346343994
testing 26 5558
ablating all relevant neurons except the tested one yields score 7.381309509277344
testing 29 4131
ablating all relevant neurons except the tested one yields score 18.551250457763672
testing 29 7261
ablating all relevant neurons except the tested one yields score 17.818199157714844


## I'm confused about the number of weakening neurons...

In [30]:
from src.weight_analysis_utils.utils import cos

In [33]:
weakening_df = subnodes_df[subnodes_df["subnode_index"].notna()].sort_values("score", ascending=False)

In [49]:
len(weakening_df)

526

In [65]:
from numpy import sign

In [ ]:
for i,row in weakening_df.iterrows():
    layer=row["layer"]
    neuron=int(row["subnode_index"])
    inout = cos(model.blocks[layer].mlp.W_in[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).item()
    gateout = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).abs().item()
    gatein = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_in[:,neuron]).abs().item()
    if inout>=-.5:
        print(layer, neuron, 'inout:', inout)
    if gateout<=.5:
        print(layer, neuron, 'abs gateout:', gateout)
    if gatein*sign(gateout)<=.5:
        print(layer, neuron, 'abs gatein:', gatein)

In [42]:
import gc
from torch.cuda import empty_cache

In [45]:
import os

In [43]:
gc.collect()
empty_cache()

In [46]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

In [47]:
model.process_weights_()

In [67]:
for i,row in weakening_df.iterrows():
    layer=row["layer"]
    neuron=int(row["subnode_index"])
    inout = cos(model.blocks[layer].mlp.W_in[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).item()
    gateout = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_out[neuron,:]).item()
    gatein = cos(model.blocks[layer].mlp.W_gate[:,neuron], model.blocks[layer].mlp.W_in[:,neuron]).item()
    if inout>=-.5:
        print(layer, neuron, 'inout:', inout)
    if abs(gateout)<=.5:
        print(layer, neuron, 'abs gateout:', gateout)
    if gatein*sign(gateout)>=-.5:
        print(layer, neuron, 'resigned gatein:', gatein)

In [52]:
from torch import load

In [ ]:
data = load(DATA_DIR + '/results/allenai/OLMo-7B-0424-hf/refactored/data.pt')

In [57]:
data.keys()

dict_keys(['d_model', 'linout', 'gatelin', 'gateout', 'beta', 'randomness', 'norm_gate', 'norm_in', 'norm_out', 'norm_in_out', 'categories', 'category_stats', 'W_gate_start_to_end', 'W_in_start_to_end', 'W_out_start_to_end'])

In [58]:
data['category_stats']

{(1,
  1,
  1): tensor([ 1,  1,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  2,  1, 15, 11,  1, 11,  1,  1,  1]),
 (1,
  1,
  0): tensor([ 0,  2,  6, 10, 14, 38, 18, 16,  9,  1,  4,  1,  2,  2, 13,  0,  1,  4,
          2,  6,  3,  5,  6,  8,  6,  6, 18,  2, 10,  0,  0,  0]),
 (1,
  0,
  1): tensor([1397, 2867, 7379, 6966, 6223, 7206, 7480, 7661, 7076, 6445, 5526, 4821,
         4279, 3926, 3876, 3768, 3420, 3312, 3488, 3692, 3030, 3305, 3075, 3090,
         2342, 2515, 1719, 1066,  404,   50,   39,  228]),
 (1,
  0,
  0): tensor([619,   3,   6,   1,   0,   2,   1,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   1,   1,   0,   0,   0,   2,   4,   7,  10,   1,
           4,   1,   7,   6]),
 (0,
  1,
  1): tensor([  34,  120,  270,  497,  601, 1022, 1156,  921,  839,  893,  710,  697,
          642,  685,  887,  923,  773,  636,  687,  866, 1155, 1162, 1215, 1183,
         1012, 1107,  997,  793,  504,   59,   92,  676])

In [60]:
data['category_stats'][(-1,1,1)]

tensor([  5,  23,   8,   5,  11,  19,   7,   6,   2,   3,   2,   7,   6,   9,
          8,   3,   4,  11,  11,  12,  11,   9,   7,  17,  15,  19,  29,  14,
         19,  36,  50, 138])

In [62]:
for key,value in data['category_stats'].items():
    print(key, value.sum())

(1, 1, 1) tensor(47)
(1, 1, 0) tensor(213)
(1, 0, 1) tensor(121671)
(1, 0, 0) tensor(676)
(0, 1, 1) tensor(23814)
(0, 1, 0) tensor(128)
(0, 0, 1) tensor(192376)
(-1, 1, 1) tensor(526)
(-1, 1, 0) tensor(321)
(-1, 0, 1) tensor(12440)
(-1, 0, 0) tensor(44)


In [68]:
model.cfg.normalization_type

'LNPre'

In [69]:
model.state_dict().keys()

odict_keys(['embed.W_E', 'blocks.0.attn.W_Q', 'blocks.0.attn.W_O', 'blocks.0.attn.b_Q', 'blocks.0.attn.b_O', 'blocks.0.attn.W_K', 'blocks.0.attn.W_V', 'blocks.0.attn.b_K', 'blocks.0.attn.b_V', 'blocks.0.attn.mask', 'blocks.0.attn.IGNORE', 'blocks.0.attn.rotary_sin', 'blocks.0.attn.rotary_cos', 'blocks.0.mlp.W_in', 'blocks.0.mlp.W_out', 'blocks.0.mlp.W_gate', 'blocks.0.mlp.b_in', 'blocks.0.mlp.b_out', 'blocks.1.attn.W_Q', 'blocks.1.attn.W_O', 'blocks.1.attn.b_Q', 'blocks.1.attn.b_O', 'blocks.1.attn.W_K', 'blocks.1.attn.W_V', 'blocks.1.attn.b_K', 'blocks.1.attn.b_V', 'blocks.1.attn.mask', 'blocks.1.attn.IGNORE', 'blocks.1.attn.rotary_sin', 'blocks.1.attn.rotary_cos', 'blocks.1.mlp.W_in', 'blocks.1.mlp.W_out', 'blocks.1.mlp.W_gate', 'blocks.1.mlp.b_in', 'blocks.1.mlp.b_out', 'blocks.2.attn.W_Q', 'blocks.2.attn.W_O', 'blocks.2.attn.b_Q', 'blocks.2.attn.b_O', 'blocks.2.attn.W_K', 'blocks.2.attn.W_V', 'blocks.2.attn.b_K', 'blocks.2.attn.b_V', 'blocks.2.attn.mask', 'blocks.2.attn.IGNORE', 'bl

In [70]:
data['category_stats'][(1,1,1)]

tensor([ 1,  1,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  2,  1, 15, 11,  1, 11,  1,  1,  1])